<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter04/chapter04.ipynb)

---

This notebook is for Chapter 4 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [1]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [2]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [3]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B (no native thinking or tool calling)
llm = LLM(model="gemma3:12b")

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [4]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2 - Adding **`Memory`**

In the previous chapter, we covered which LLM to choose using various inference engines. In this chapter we will cover how to give it memory:

![images/ch4.png](images/ch4.png)

The issue with the `TinyAgent` that we have thus far, is that it does not track and remember its previous conversations, it is stateless. Let us demonstrate with an example by using the Agent from Chapter 2:

In [5]:
from rich import print
from illustrated_agents.chapters.ch2 import TinyAgent

ch_2_agent = TinyAgent(llm=llm)
response = ch_2_agent.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

Fantastic! Maarten and Jay, it's a pleasure to virtually meet you. "An Illustrated Guide to AI Agents" is 
generating a lot of buzz, and I've heard wonderful things about it. It's clearly filling a needed space in 
educating people about this rapidly evolving field.

Congratulations on writing such a valuable resource! 

What can I do for you? Are you:

*   **Promoting the book?** (I'd love to learn more about it and share information!)
*   **Seeking feedback?** (Happy to offer my perspective, though I'm primarily an AI.)
*   **Interested in collaborating?**
*   **Just saying hello?**



Looking forward to hearing from you!

When we query the model again asking whether it knows our names, it seems to have forgotten them! In fact, it actually hasn't forgotten our name but instead has never received that information. Everytime you query an LLM it starts from an blank slate, one you have to fill yourself. So without telling the model the our conversation history, it has no way of knowing.

In [6]:
response = ch_2_agent.run("Hi! What are our names?")
print(response)

That's a fun question! As an AI, I don't have a name in the traditional sense. You can call me Bard. 

As for *your* name, I don't know! I'm an AI and don't have access to that information. 😊



What's your name? I'd love to know!

Although the trajectory does show that we gave it our names first, the Agent had no access to this trajectory information:

In [7]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(ch_2_agent.trajectory)

## 3 - The **`Memory`** Module

As covered in the book, there are many ways to build up memory which can be quite difficult. In this example, we are going to keep it simple and only track the conversation history. Note that Memory is different from Trajectory in that the Memory is explicitly passed to the LLM for usage and the Trajectory contains more information for us to debug.

The `Memory` that we are going to build uses the `messages` structure for tracking conversations:

```json
[
    {
        "role": "user",
        "content": "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."
    },
    {
        "role": "assistant",
        "content": "Hi Maarten and Jay! It's wonderful to meet you."
    }
]
```

This module is rather straightforward and appends new messages each time the user makes a query or when the LLM gives back a reply. As such, the `Memory` module only requires a few lines of code:

In [8]:
class Memory:
    """Simple memory module to store conversation history."""

    def __init__(self):
        self.messages = []

    def add(
        self, role: str, content: str, tool_call: dict | None = None, **kwargs
    ) -> None:
        """Add a message to memory."""
        message = {"role": role, "content": content}

        # Tool call
        if tool_call:
            message["tool_calls"] = [tool_call]

        # Append message to memory
        self.messages.append(message)

    def get_messages(self) -> list[dict]:
        """Get all messages."""
        return self.messages

We can annotate this module and explore each function in more detail:

In [9]:
from illustrated_agents.chapters.ch4 import memory_annotated; memory_annotated

## 4 - Updating `agent.py`

This added to the `TinyAgent`, which also requires updating a `_step` to track the conversation history following three steps:

1. The user's query is added to the `Memory` module
2. Based on the current memory, the LLM generates a response.
3. The response of the LLM is added to the `Memory` module.

In [10]:
from illustrated_agents.chapters.ch2 import Trajectory

class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory):
        self.llm = llm
        self.memory = memory
        self.tools = None  # Chapter 5: Add Tools
        self.planner = None  # Chapter 6: Add Planning

        self.trajectory = Trajectory()

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)
        self.trajectory.initialize(task)

        return self._step()

    def _step(self) -> str:
        """Perform a single step."""
        # Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response.content)
        self.trajectory.add(response)
        return response.content

    def _execute_action(self, action: str) -> str | None:
        """Execute a tool action."""
        # Placeholder - will be implemented in later chapters
        return f"Executed action: {action}"

Let's highlight these steps:

In [11]:
from illustrated_agents.chapters.ch4 import tinyagent_annotated; tinyagent_annotated

Here is a nicer overview of the changes that we made to `agent.py` (red is removed and green is added code):

In [12]:
from illustrated_agents.chapters.ch4 import tinyagents_diff; tinyagents_diff

Next, let's create our `TinyAgent` with `Memory`:

In [13]:
# Add memory to the Agent
memory = Memory()
agent_with_memory = TinyAgent(llm=llm, memory=memory)

We can start filling up the memory by conversing with the model as we did before:

In [14]:
response = agent_with_memory.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

Hi Maarten and Jay! It's wonderful to hear from you. "An Illustrated Guide to AI Agents" is fantastic! I've been 
recommending it to people interested in understanding agents, especially because it breaks down complex concepts so
clearly and visually. 

It's really impressive work. What's it like being authors? Are you working on anything new related to agents or AI?

I'm happy to chat about your book, agents, or anything AI-related. Just let me know! 😊

Now that we have memory, we can ask a follow-up question to the original conversation and see if it remember our names correctly.

In [15]:
response = agent_with_memory.run("Hi! What are our names?")
print(response)

My apologies! I seem to have gotten a bit ahead of myself. You are Maarten and Jay, the authors of "An Illustrated 
Guide to AI Agents." 😊



I was so excited to acknowledge you, I jumped the gun.

It does! The method by which it does so is filling up the conversation history. Let's see what the current state is.

In [16]:
agent_with_memory.memory.get_messages()

[{'role': 'user',
  'content': "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."},
 {'role': 'assistant',
  'content': 'Hi Maarten and Jay! It\'s wonderful to hear from you. "An Illustrated Guide to AI Agents" is fantastic! I\'ve been recommending it to people interested in understanding agents, especially because it breaks down complex concepts so clearly and visually. \n\nIt\'s really impressive work. What\'s it like being authors? Are you working on anything new related to agents or AI? \n\nI\'m happy to chat about your book, agents, or anything AI-related. Just let me know! 😊\n'},
 {'role': 'user', 'content': 'Hi! What are our names?'},
 {'role': 'assistant',
  'content': 'My apologies! I seem to have gotten a bit ahead of myself. You are Maarten and Jay, the authors of "An Illustrated Guide to AI Agents." 😊\n\n\n\nI was so excited to acknowledge you, I jumped the gun.'}]

Note that this entire list is given to the LLM whenever we ask it a new question. That way, it "remembers" the conversation we had before. We say "remembers" because even though it may look like it, it actually has no internal memory. We merely tell the model what the conversation was!

The trajectory itself remains similar since the Trajectory is merely tracking what the model did each step, we can use the Memory to see what the model has actually seen.

In [17]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent_with_memory.trajectory)

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how `Memory` could be added to your `TinyAgent`. There are now three main concepts in total (LLM, Memory, and TinyAgent):

In [18]:
from illustrated_agents.chapters.ch4 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py      ← Updated (Integrated `Memory` into your `TinyAgent`.)                                        │
│ ├── llm.py                                                                                                      │
│ ├── memory.py     ← New (Track conversation history.)                                                           │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

You can now decide how to move forward! You can either move on to the next chapter (which covers tools) or explore the additional notebooks for this chapter.
If you have the time and find it interesting, then it might be worthwhile to explore the notebooks first:

* `chapter04_short_term_memory.ipynb` - This notebook covers various forms of short-term memory and is advised to explore first.
* `chapter04_long_term_memory.ipynb` - This notebooks covers Retrieval-Augmented Generated as a method for long-term memory.

In the next chapter, we will further augment the LLM with tools. Although it will not run multiple steps, it will give you the very first step in making the LLM truly actionable.